# Phase 4.3 — Route-Month Data Foundation

## Phase 4.3.1 — Raw T-100 Grain & Route-Month Multiplicity Investigation

Purpose: investigate why multiple legitimate T-100 Segment records can exist
for the same WN directional route-month before defining aggregation rules.

In [1]:
from pathlib import Path
import pandas as pd

In [2]:
raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2024.zip")
df_2024 = pd.read_csv(raw_path)
raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2025.zip")
df_2025 = pd.read_csv(raw_path)
raw_path = Path("../data/raw/T_T100_SEGMENT_ALL_CARRIER_2026.zip")
df_2026 = pd.read_csv(raw_path)

In [3]:
df_historical = pd.concat([df_2024, df_2025, df_2026], ignore_index=True)
print("Shape of df_historical:", df_historical.shape)
df_historical.head()

Shape of df_historical: (1356516, 50)


,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,PAYLOAD,SEATS,PASSENGERS,FREIGHT,MAIL,DISTANCE,RAMP_TO_RAMP,AIR_TIME,...,DEST_WAC,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,YEAR,QUARTER,MONTH,DISTANCE_GROUP,CLASS,DATA_SOURCE
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,...,41,6,622,1,2024,4,10,1,L,DU
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,1,1,1,F,DU
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,3,7,1,F,DU
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,3,8,1,F,DU
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,0.0,0.0,...,15,0,79,1,2024,4,12,1,F,DU


In [4]:
del df_2024, df_2025, df_2026

In [5]:
wn_f_historical = df_historical[
    (df_historical['UNIQUE_CARRIER'] == 'WN') &
     (df_historical['CLASS'] == 'F') 
        ].copy()

print("shape of wn_f_historical:", wn_f_historical.shape)
wn_f_historical.head()

shape of wn_f_historical: (139082, 50)


,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED,PAYLOAD,SEATS,PASSENGERS,FREIGHT,MAIL,DISTANCE,RAMP_TO_RAMP,AIR_TIME,...,DEST_WAC,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,YEAR,QUARTER,MONTH,DISTANCE_GROUP,CLASS,DATA_SOURCE
37979,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,29.0,29.0,9.0,...,91,6,612,1,2024,4,10,1,F,DU
37980,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,104.0,66.0,24.0,...,33,6,612,1,2024,1,1,1,F,DU
37981,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,109.0,93.0,34.0,...,91,6,612,1,2024,1,2,1,F,DU
37982,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,229.0,49.0,39.0,...,42,6,612,1,2024,3,8,1,F,DU
37983,0.0,1.0,34600.0,143.0,0.0,0.0,0.0,241.0,51.0,42.0,...,22,6,612,1,2024,1,3,1,F,DU


In [6]:
wn_f_historical[['YEAR', 'MONTH']].drop_duplicates().sort_values(['YEAR', 'MONTH']).reset_index(drop=True)

,YEAR,MONTH
0,2024,1
1,2024,2
2,2024,3
3,2024,4
4,2024,5
5,2024,6
6,2024,7
7,2024,8
8,2024,9
9,2024,10


In [7]:
route_month_row_counts  = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])
    .size()
    .reset_index(name = 'RAW_ROW_COUNT')
    )

print(route_month_row_counts.head(10))
print(route_month_row_counts.shape)

   YEAR  MONTH ORIGIN DEST  RAW_ROW_COUNT
0  2024      1    ABQ  AUS              3
1  2024      1    ABQ  BUR              1
2  2024      1    ABQ  BWI              3
3  2024      1    ABQ  DAL              3
4  2024      1    ABQ  DEN              3
5  2024      1    ABQ  HOU              3
6  2024      1    ABQ  LAS              3
7  2024      1    ABQ  LAX              3
8  2024      1    ABQ  LGB              3
9  2024      1    ABQ  MCI              1
(55395, 5)


In [8]:
route_month_row_counts["RAW_ROW_COUNT"].value_counts().sort_index()

RAW_ROW_COUNT
1    10419
2     6464
3    38342
4      144
5       23
6        3
Name: count, dtype: int64

In [9]:
print('single_row_route_months: ',len(route_month_row_counts[route_month_row_counts["RAW_ROW_COUNT"] == 1]))
multi_row_route_months = route_month_row_counts[route_month_row_counts["RAW_ROW_COUNT"] > 1]
print('multi_row_route_months: ',len(multi_row_route_months))
print('multi_row_route_months percentage: ',len(multi_row_route_months)/len(route_month_row_counts)*100)


single_row_route_months:  10419
multi_row_route_months:  44976
multi_row_route_months percentage:  81.19144327105334


In [10]:
route_month_row_counts[
    route_month_row_counts["RAW_ROW_COUNT"] == 2
].head()


,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT
10,2024,1,ABQ,MCO,2
24,2024,1,AMA,AUS,2
44,2024,1,ATL,IAD,2
83,2024,1,AUS,CUN,2
121,2024,1,BDL,DCA,2


In [11]:
route_month_row_counts[
    route_month_row_counts["RAW_ROW_COUNT"] == 6
].head()

,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT
31815,2025,5,SLC,DEN,6
37035,2025,8,FLL,MCO,6
50609,2026,3,MCO,BWI,6


In [12]:
abq_mco_2024_01 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 1) &
    (wn_f_historical["ORIGIN"] == "ABQ") &
    (wn_f_historical["DEST"] == "MCO"),
    [
        "UNIQUE_CARRIER",
        "YEAR",
        "MONTH",
        "ORIGIN",
        "DEST",
        "AIRCRAFT_GROUP",
        "AIRCRAFT_TYPE",
        "AIRCRAFT_CONFIG",
        "UNIQUE_CARRIER_ENTITY",
        "DATA_SOURCE",
        "DISTANCE",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED"
    ]
].sort_values("AIRCRAFT_TYPE").reset_index(drop=True)

abq_mco_2024_01

,UNIQUE_CARRIER,YEAR,MONTH,ORIGIN,DEST,AIRCRAFT_GROUP,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,UNIQUE_CARRIER_ENTITY,DATA_SOURCE,DISTANCE,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
0,WN,2024,1,ABQ,MCO,6,612,1,06725,DU,1553.0,402.0,429.0,3.0,3.0
1,WN,2024,1,ABQ,MCO,6,614,1,06725,DU,1553.0,145.0,175.0,1.0,1.0


In [104]:
aircraft_type_nunique = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])["AIRCRAFT_TYPE"]
    .nunique()
    .reset_index(name="AIRCRAFT_TYPE_NUNIQUE")
)
aircraft_type_nunique.head(10)

,YEAR,MONTH,ORIGIN,DEST,AIRCRAFT_TYPE_NUNIQUE
0,2024,1,ABQ,AUS,3
1,2024,1,ABQ,BUR,1
2,2024,1,ABQ,BWI,3
3,2024,1,ABQ,DAL,3
4,2024,1,ABQ,DEN,3
5,2024,1,ABQ,HOU,3
6,2024,1,ABQ,LAS,3
7,2024,1,ABQ,LAX,3
8,2024,1,ABQ,LGB,3
9,2024,1,ABQ,MCI,1


In [15]:
route_month_row_counts = route_month_row_counts.merge(
    aircraft_type_nunique,
    how="inner",
    on=["YEAR", "MONTH", "ORIGIN", "DEST"]
)



In [16]:
single_type_multi_row = route_month_row_counts[
    (route_month_row_counts["RAW_ROW_COUNT"] > 1) &
    (route_month_row_counts["AIRCRAFT_TYPE_NUNIQUE"] == 1)
]
print(len(single_type_multi_row))
single_type_multi_row


2


,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT,AIRCRAFT_TYPE_NUNIQUE
28134,2025,3,TPA,MCO,2,1
33217,2025,6,MCO,TPA,2,1


In [17]:
tpa_mco_2025_03 = wn_f_historical.loc[
    (wn_f_historical["ORIGIN"] == "TPA") &
    (wn_f_historical["DEST"] == "MCO") &
    (wn_f_historical["MONTH"] == 3) &
    (wn_f_historical["YEAR"] == 2025),
    [
    "UNIQUE_CARRIER",
    "UNIQUE_CARRIER_NAME",
    "CARRIER",
    "CARRIER_NAME",
    "UNIQUE_CARRIER_ENTITY",
    "REGION",
    "AIRCRAFT_TYPE",
    "AIRCRAFT_CONFIG",
    "DATA_SOURCE",
    "DISTANCE",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"
]
]

tpa_mco_2025_03

,UNIQUE_CARRIER,UNIQUE_CARRIER_NAME,CARRIER,CARRIER_NAME,UNIQUE_CARRIER_ENTITY,REGION,AIRCRAFT_TYPE,AIRCRAFT_CONFIG,DATA_SOURCE,DISTANCE,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
599578,WN,Southwest Airlines Co.,WN,Southwest Airlines Co.,06725,D,838,1,DU,81.0,77.0,175.0,0.0,1.0
599946,WN,Southwest Airlines Co.,WN,Southwest Airlines Co.,11033,L,838,1,DU,81.0,156.0,175.0,0.0,1.0


In [18]:
mco_tpa_2025_06 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2025) &
    (wn_f_historical["MONTH"] == 6) &
    (wn_f_historical["ORIGIN"] == "MCO") &
    (wn_f_historical["DEST"] == "TPA"),
    [
        "UNIQUE_CARRIER_ENTITY",
        "REGION",
        "AIRCRAFT_GROUP",
        "AIRCRAFT_TYPE",
        "AIRCRAFT_CONFIG",
        "DATA_SOURCE",
        "DISTANCE",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED"
    ]
]

comparison = mco_tpa_2025_06.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2"]

different_fields

,ROW_1,ROW_2
UNIQUE_CARRIER_ENTITY,11033,06725
REGION,L,D
PASSENGERS,25.0,284.0
SEATS,175.0,350.0
DEPARTURES_PERFORMED,1.0,2.0


In [19]:
entity_nunique = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])["UNIQUE_CARRIER_ENTITY"]
    .nunique()
    .reset_index(name="ENTITY_NUNIQUE")
)
entity_nunique.head(10)

,YEAR,MONTH,ORIGIN,DEST,ENTITY_NUNIQUE
0,2024,1,ABQ,AUS,1
1,2024,1,ABQ,BUR,1
2,2024,1,ABQ,BWI,1
3,2024,1,ABQ,DAL,1
4,2024,1,ABQ,DEN,1
5,2024,1,ABQ,HOU,1
6,2024,1,ABQ,LAS,1
7,2024,1,ABQ,LAX,1
8,2024,1,ABQ,LGB,1
9,2024,1,ABQ,MCI,1


In [20]:
entity_nunique["ENTITY_NUNIQUE"].value_counts().sort_index()

ENTITY_NUNIQUE
1    55205
2      190
Name: count, dtype: int64

In [21]:
route_month_entity_summary = route_month_row_counts.merge(
    entity_nunique,
    how="inner",
    on=["YEAR", "MONTH", "ORIGIN", "DEST"]
)
multi_entity_route_months = route_month_entity_summary[
    route_month_entity_summary["ENTITY_NUNIQUE"] > 1
]
multi_entity_route_months["RAW_ROW_COUNT"].value_counts().sort_index()

RAW_ROW_COUNT
2      7
3     13
4    144
5     23
6      3
Name: count, dtype: int64

In [22]:
route_month_dimension_summary = (
    route_month_row_counts
    .merge(
        aircraft_type_nunique,
        on=["YEAR", "MONTH", "ORIGIN", "DEST"],
        how="inner"
    )
    .merge(
        entity_nunique,
        on=["YEAR", "MONTH", "ORIGIN", "DEST"],
        how="inner"
    )
)
route_month_dimension_summary = route_month_dimension_summary.rename(columns={
    "AIRCRAFT_TYPE_NUNIQUE_x": "AIRCRAFT_TYPE_NUNIQUE"})
route_month_dimension_summary = route_month_dimension_summary.drop(columns=["AIRCRAFT_TYPE_NUNIQUE_y"])

In [23]:
two_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 2
]
two_row_route_months

,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT,AIRCRAFT_TYPE_NUNIQUE,ENTITY_NUNIQUE
10,2024,1,ABQ,MCO,2,2,1
24,2024,1,AMA,AUS,2,2,1
44,2024,1,ATL,IAD,2,2,1
83,2024,1,AUS,CUN,2,2,1
121,2024,1,BDL,DCA,2,2,1
...,...,...,...,...,...,...,...
55363,2026,5,TUL,AUS,2,2,1
55387,2026,5,TYS,MCO,2,2,1
55389,2026,5,VPS,BWI,2,2,1
55393,2026,5,VPS,MCI,2,2,1


In [24]:
two_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
1                      2                    2
2                      1                 6457
                       2                    5
dtype: int64

In [25]:
mixed_two_row = two_row_route_months[
    (two_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 2) &
    (two_row_route_months["ENTITY_NUNIQUE"] == 2)
]

mixed_two_row

,YEAR,MONTH,ORIGIN,DEST,RAW_ROW_COUNT,AIRCRAFT_TYPE_NUNIQUE,ENTITY_NUNIQUE
14968,2024,8,PBI,MCO,2,2,2
15181,2024,8,RSW,MCO,2,2,2
15224,2024,8,SAT,AUS,2,2,2
30874,2025,5,IAD,BWI,2,2,2
39143,2025,9,JAX,MCO,2,2,2


In [26]:
pbi_mco_2024_08 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 8) &
    (wn_f_historical["ORIGIN"] == "PBI") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = pbi_mco_2024_08.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2"]

different_fields

,ROW_1,ROW_2
PAYLOAD,34600.0,43400.0
SEATS,143.0,175.0
PASSENGERS,137.0,173.0
RAMP_TO_RAMP,54.0,56.0
AIR_TIME,31.0,38.0
UNIQUE_CARRIER_ENTITY,06725,11033
REGION,D,L
AIRCRAFT_TYPE,612,838


In [27]:
three_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 3
]

three_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
2                      2                     8
3                      1                 38329
                       2                     5
dtype: int64

In [28]:
example_3row_2type_2entity = three_row_route_months[
    (three_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 2) &
    (three_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_3row_2type_2entity

YEAR                     2024
MONTH                       6
ORIGIN                    PBI
DEST                      MCO
RAW_ROW_COUNT               3
AIRCRAFT_TYPE_NUNIQUE       2
ENTITY_NUNIQUE              2
Name: 10850, dtype: object

In [29]:
pbi_mco_2024_06 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 6) &
    (wn_f_historical["ORIGIN"] == "PBI") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = pbi_mco_2024_06.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3"]

different_fields

,ROW_1,ROW_2,ROW_3
DEPARTURES_PERFORMED,1.0,1.0,2.0
PAYLOAD,34600.0,43400.0,86800.0
SEATS,143.0,175.0,350.0
PASSENGERS,143.0,175.0,347.0
RAMP_TO_RAMP,55.0,67.0,178.0
AIR_TIME,36.0,31.0,63.0
UNIQUE_CARRIER_ENTITY,06725,11033,06725
REGION,D,L,D
AIRCRAFT_TYPE,612,838,838


In [30]:
example_3row_3type_2entity = three_row_route_months[
    (three_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (three_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_3row_3type_2entity

YEAR                     2024
MONTH                       6
ORIGIN                    AMA
DEST                      DEN
RAW_ROW_COUNT               3
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 9399, dtype: object

In [31]:
ama_den_2024_06 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 6) &
    (wn_f_historical["ORIGIN"] == "AMA") &
    (wn_f_historical["DEST"] == "DEN")
 
]

comparison = ama_den_2024_06.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3"]

different_fields

,ROW_1,ROW_2,ROW_3
DEPARTURES_SCHEDULED,0.0,0.0,1.0
DEPARTURES_PERFORMED,1.0,1.0,2.0
PAYLOAD,43400.0,43400.0,69200.0
SEATS,175.0,175.0,286.0
PASSENGERS,158.0,174.0,214.0
FREIGHT,0.0,159.0,0.0
RAMP_TO_RAMP,76.0,65.0,133.0
AIR_TIME,55.0,56.0,110.0
UNIQUE_CARRIER_ENTITY,11033,06725,06725
REGION,L,D,D


In [32]:
four_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 4
]

four_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
2                      2                   1
3                      2                 143
dtype: int64

In [33]:
example_4row_2type_2entity = four_row_route_months[
    (four_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 2) &
    (four_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_4row_2type_2entity

YEAR                     2025
MONTH                       8
ORIGIN                    TPA
DEST                      MCO
RAW_ROW_COUNT               4
AIRCRAFT_TYPE_NUNIQUE       2
ENTITY_NUNIQUE              2
Name: 38333, dtype: object

In [34]:
tpa_mco_2025_08 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2025) &
    (wn_f_historical["MONTH"] == 8) &
    (wn_f_historical["ORIGIN"] == "TPA") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = tpa_mco_2025_08.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4
DEPARTURES_PERFORMED,1.0,1.0,3.0,5.0
PAYLOAD,34600.0,43400.0,130200.0,173000.0
SEATS,143.0,175.0,525.0,715.0
PASSENGERS,26.0,146.0,350.0,627.0
FREIGHT,0.0,0.0,664.0,1283.0
RAMP_TO_RAMP,39.0,54.0,152.0,226.0
AIR_TIME,22.0,21.0,71.0,112.0
UNIQUE_CARRIER_ENTITY,11033,11033,06725,06725
REGION,L,L,D,D
AIRCRAFT_TYPE,612,838,838,612


In [35]:
example_4row_3type_2entity = four_row_route_months[
    (four_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (four_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_4row_3type_2entity

YEAR                     2024
MONTH                       1
ORIGIN                    BWI
DEST                      MCO
RAW_ROW_COUNT               4
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 273, dtype: object

In [36]:
bwi_mco_2024_01 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 1) &
    (wn_f_historical["ORIGIN"] == "BWI") &
    (wn_f_historical["DEST"] == "MCO")
 
]

comparison = bwi_mco_2024_01.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4
DEPARTURES_SCHEDULED,0.0,87.0,104.0,119.0
DEPARTURES_PERFORMED,1.0,84.0,103.0,114.0
PAYLOAD,34600.0,2906400.0,4470200.0,4947600.0
SEATS,143.0,12012.0,18025.0,19950.0
PASSENGERS,143.0,9392.0,13679.0,15683.0
FREIGHT,0.0,8983.0,18899.0,29860.0
RAMP_TO_RAMP,171.0,12486.0,15055.0,16617.0
AIR_TIME,142.0,10355.0,12414.0,13849.0
UNIQUE_CARRIER_ENTITY,11033,06725,06725,06725
REGION,L,D,D,D


In [37]:
five_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 5
]

five_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
3                      2                 23
dtype: int64

In [38]:
example_5row_3type_2entity = five_row_route_months[
    (five_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (five_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_5row_3type_2entity

YEAR                     2024
MONTH                       1
ORIGIN                    AUS
DEST                      HOU
RAW_ROW_COUNT               5
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 89, dtype: object

In [39]:
aus_hou_2024_01 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2024) &
    (wn_f_historical["MONTH"] == 1) &
    (wn_f_historical["ORIGIN"] == "AUS") &
    (wn_f_historical["DEST"] == "HOU")
 
]

comparison = aus_hou_2024_01.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4", "ROW_5"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4,ROW_5
DEPARTURES_SCHEDULED,0.0,0.0,27.0,41.0,83.0
DEPARTURES_PERFORMED,1.0,2.0,26.0,40.0,86.0
PAYLOAD,43400.0,69200.0,1128400.0,1736000.0,2975600.0
SEATS,175.0,286.0,4550.0,7000.0,12298.0
PASSENGERS,168.0,282.0,2167.0,3292.0,6012.0
FREIGHT,0.0,0.0,6558.0,5812.0,16600.0
RAMP_TO_RAMP,125.0,144.0,1307.0,2219.0,4747.0
AIR_TIME,46.0,75.0,817.0,1281.0,2759.0
UNIQUE_CARRIER_ENTITY,11033,11033,06725,06725,06725
REGION,L,L,D,D,D


In [40]:
six_row_route_months = route_month_dimension_summary[
    route_month_dimension_summary["RAW_ROW_COUNT"] == 6
]

six_row_route_months.groupby(
    ["AIRCRAFT_TYPE_NUNIQUE", "ENTITY_NUNIQUE"]
).size()

AIRCRAFT_TYPE_NUNIQUE  ENTITY_NUNIQUE
3                      2                 3
dtype: int64

In [41]:
example_6row_3type_2entity = six_row_route_months[
    (six_row_route_months["AIRCRAFT_TYPE_NUNIQUE"] == 3) &
    (six_row_route_months["ENTITY_NUNIQUE"] == 2)
].iloc[0]

example_6row_3type_2entity

YEAR                     2025
MONTH                       5
ORIGIN                    SLC
DEST                      DEN
RAW_ROW_COUNT               6
AIRCRAFT_TYPE_NUNIQUE       3
ENTITY_NUNIQUE              2
Name: 31815, dtype: object

In [42]:
slc_den_2025_05 = wn_f_historical.loc[
    (wn_f_historical["YEAR"] == 2025) &
    (wn_f_historical["MONTH"] == 5) &
    (wn_f_historical["ORIGIN"] == "SLC") &
    (wn_f_historical["DEST"] == "DEN")
 
]

comparison = slc_den_2025_05.reset_index(drop=True).T

different_fields = comparison[
    comparison.nunique(axis=1, dropna=False) > 1
]

different_fields.columns = ["ROW_1", "ROW_2", "ROW_3", "ROW_4", "ROW_5", "ROW_6"]

different_fields

,ROW_1,ROW_2,ROW_3,ROW_4,ROW_5,ROW_6
DEPARTURES_SCHEDULED,0.0,0.0,0.0,50.0,55.0,84.0
DEPARTURES_PERFORMED,1.0,1.0,1.0,56.0,56.0,87.0
PAYLOAD,34600.0,43400.0,43400.0,2430400.0,1937600.0,3775800.0
SEATS,143.0,175.0,175.0,9800.0,8008.0,15225.0
PASSENGERS,100.0,152.0,173.0,7394.0,6706.0,12542.0
FREIGHT,0.0,0.0,0.0,24249.0,25593.0,44397.0
RAMP_TO_RAMP,86.0,76.0,89.0,4978.0,4925.0,7587.0
AIR_TIME,62.0,60.0,57.0,3563.0,3537.0,5493.0
UNIQUE_CARRIER_ENTITY,11033,11033,11033,06725,06725,06725
REGION,L,L,L,D,D,D


In [43]:
grain_field_variation = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])
    .agg(
        AIRCRAFT_CONFIG_NUNIQUE=("AIRCRAFT_CONFIG", "nunique"),
        AIRCRAFT_GROUP_NUNIQUE=("AIRCRAFT_GROUP", "nunique"),
        DATA_SOURCE_NUNIQUE=("DATA_SOURCE", "nunique"),
        DISTANCE_NUNIQUE=("DISTANCE", "nunique")
    )
)

grain_field_variation.apply(lambda col: col.value_counts().sort_index())

,AIRCRAFT_CONFIG_NUNIQUE,AIRCRAFT_GROUP_NUNIQUE,DATA_SOURCE_NUNIQUE,DISTANCE_NUNIQUE
1,55395,55395,55395,55395


In [44]:
activity_variation = (
    wn_f_historical
    .groupby(["YEAR", "MONTH", "ORIGIN", "DEST"])
    .agg(
        RAW_ROW_COUNT=("PASSENGERS", "size"),
        PASSENGERS_NUNIQUE=("PASSENGERS", "nunique"),
        SEATS_NUNIQUE=("SEATS", "nunique"),
        DEPARTURES_SCHEDULED_NUNIQUE=("DEPARTURES_SCHEDULED", "nunique"),
        DEPARTURES_PERFORMED_NUNIQUE=("DEPARTURES_PERFORMED", "nunique")
    )
)

multi_row_activity_variation = activity_variation[
    activity_variation["RAW_ROW_COUNT"] > 1
]

(
    multi_row_activity_variation[
        [
            "PASSENGERS_NUNIQUE",
            "SEATS_NUNIQUE",
            "DEPARTURES_SCHEDULED_NUNIQUE",
            "DEPARTURES_PERFORMED_NUNIQUE"
        ]
    ] > 1
).sum()

PASSENGERS_NUNIQUE              44959
SEATS_NUNIQUE                   44455
DEPARTURES_SCHEDULED_NUNIQUE    43822
DEPARTURES_PERFORMED_NUNIQUE    43991
dtype: int64

In [45]:
wn_f_historical.duplicated().sum()

np.int64(0)

In [46]:
grain_columns = [
    "YEAR",
    "MONTH",
    "ORIGIN",
    "DEST",
    "AIRCRAFT_TYPE",
    "UNIQUE_CARRIER_ENTITY",
    "REGION",
    "AIRCRAFT_CONFIG",
    "AIRCRAFT_GROUP",
    "DATA_SOURCE",
    "DISTANCE"
]

wn_f_historical[grain_columns].isna().sum()

YEAR                     0
MONTH                    0
ORIGIN                   0
DEST                     0
AIRCRAFT_TYPE            0
UNIQUE_CARRIER_ENTITY    0
REGION                   0
AIRCRAFT_CONFIG          0
AIRCRAFT_GROUP           0
DATA_SOURCE              0
DISTANCE                 0
dtype: int64

### Phase 4.3.1 — Raw Grain Investigation Conclusions

- The working WN scheduled-service dataset contains 139,082 raw T-100 Segment rows covering 55,395 apparent directional route-months from January 2024 through May 2026.
- A raw T-100 Segment row is not equivalent to one analytical route-month. About 81% of apparent route-months contain multiple legitimate source rows.
- Route-month multiplicity is primarily explained by `AIRCRAFT_TYPE` and, in a smaller number of cases, `UNIQUE_CARRIER_ENTITY` / `REGION`.
- Different entity × aircraft-type combinations appear only when corresponding activity is reported; not every theoretical combination must exist.
- `AIRCRAFT_CONFIG`, `AIRCRAFT_GROUP`, `DATA_SOURCE`, and `DISTANCE` do not vary within the apparent route-month grain and therefore do not explain multiplicity.
- `PASSENGERS`, `SEATS`, `DEPARTURES_SCHEDULED`, and `DEPARTURES_PERFORMED` usually differ across legitimate raw rows within multi-row route-months, indicating that the rows represent separate portions of reported activity rather than duplicate route-month totals.
- No exact duplicate raw rows were found.
- No null values were found in the route-month key or grain-explanation fields inspected.
- Some inspected raw records contain performed departures even when scheduled departures are zero. These records are retained unchanged during Phase 4.3.1.
- No final route-month aggregation rules are defined in this phase.

## Phase 4.3.2 — Route-Month Aggregation Rules & Analytical Schema

### Candidate field classification

Before aggregating the raw T-100 records, candidate fields were classified according to their meaning at the directional route-month grain.

**Additive measures**
- PASSENGERS
- SEATS
- DEPARTURES_SCHEDULED
- DEPARTURES_PERFORMED

**Validated constants / identifiers**
- UNIQUE_CARRIER
- AIRLINE_ID
- UNIQUE_CARRIER_NAME
- YEAR
- MONTH
- ORIGIN
- DEST
- DISTANCE

**Derived temporal field**
- YEAR_MONTH

**Raw-grain/source dimensions not intended to survive directly in the base route-month schema**
- AIRCRAFT_TYPE
- UNIQUE_CARRIER_ENTITY
- REGION
- AIRCRAFT_GROUP
- AIRCRAFT_CONFIG
- DATA_SOURCE

In [47]:
wn_f_historical[
    (wn_f_historical['YEAR'] == 2024) &
    (wn_f_historical['MONTH'] == 1) &
    (wn_f_historical['ORIGIN'] == 'ABQ') &
    (wn_f_historical['DEST'] == 'MCO')][
        [
            'AIRCRAFT_TYPE',
            'UNIQUE_CARRIER_ENTITY',
            'PASSENGERS',
            'SEATS',
            'DEPARTURES_SCHEDULED',
            'DEPARTURES_PERFORMED'
        ]
    ]

,AIRCRAFT_TYPE,UNIQUE_CARRIER_ENTITY,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
194235,614,06725,145.0,175.0,1.0,1.0
245123,612,06725,402.0,429.0,3.0,3.0


In [48]:
distance_nunique = wn_f_historical[[
    'UNIQUE_CARRIER',
    'YEAR',
    'MONTH',
    'ORIGIN',
    'DEST',
    'DISTANCE'
]].groupby(
    ['UNIQUE_CARRIER', 'YEAR', 'MONTH', 'ORIGIN', 'DEST'])['DISTANCE'].nunique()
distance_nunique[distance_nunique > 1]

Series([], Name: DISTANCE, dtype: int64)

In [49]:
wn_f_historical[
    [
        'UNIQUE_CARRIER',
        'AIRLINE_ID',
        'UNIQUE_CARRIER_NAME'
    ]
].drop_duplicates()

,UNIQUE_CARRIER,AIRLINE_ID,UNIQUE_CARRIER_NAME
37979,WN,19393,Southwest Airlines Co.


### Route-month aggregation rules and proposed analytical schema

The raw WN `CLASS = "F"` T-100 Segment data can contain multiple legitimate records for the same directional route-month. Phase 4.3.1 showed that this multiplicity is mainly caused by aircraft type and, in some cases, reporting entity differences.

For Phase 4.3.2, each candidate field is treated according to its meaning at the directional route-month grain.

#### Additive measures

The following fields represent separate portions of operational activity across legitimate raw component rows and should therefore be summed within each directional route-month:

- `PASSENGERS`
- `SEATS`
- `DEPARTURES_SCHEDULED`
- `DEPARTURES_PERFORMED`

For example, WN ABQ → MCO in January 2024 contains separate aircraft-type records with different passenger and seat counts. These values represent separate portions of the month's activity, so they must be added when the records are collapsed to one route-month observation.

#### Validated constants and identifiers

The following fields should have one valid value within each analytical route-month:

- `UNIQUE_CARRIER`
- `AIRLINE_ID`
- `UNIQUE_CARRIER_NAME`
- `YEAR`
- `MONTH`
- `ORIGIN`
- `DEST`
- `DISTANCE`

The carrier identifiers were verified to be consistent across the selected WN dataset:

- `UNIQUE_CARRIER = "WN"`
- `AIRLINE_ID = 19393`
- `UNIQUE_CARRIER_NAME = "Southwest Airlines Co."`

`DISTANCE` was also validated across the full WN `CLASS = "F"` dataset. No route-month had more than one distinct distance value, so one validated distance value can be preserved for each route-month.

#### Derived temporal field

`YEAR_MONTH` will be created from `YEAR` and `MONTH` after aggregation and used as the monthly temporal identifier.

#### Raw-grain/source dimensions omitted from the base schema

The following fields will not be retained directly in the base route-month analytical dataset:

- `AIRCRAFT_TYPE`
- `UNIQUE_CARRIER_ENTITY`
- `REGION`
- `AIRCRAFT_GROUP`
- `AIRCRAFT_CONFIG`
- `DATA_SOURCE`

`AIRCRAFT_TYPE`, `UNIQUE_CARRIER_ENTITY`, and `REGION` can vary across legitimate raw rows within the same route-month, so retaining one arbitrary value would be misleading.

`AIRCRAFT_GROUP`, `AIRCRAFT_CONFIG`, and `DATA_SOURCE` were constant within route-months during the Phase 4.3.1 investigation, but they are not required for route-month identity, the core operational totals, or the compact analytical foundation. They are therefore intentionally omitted from the base schema.

#### Proposed route-month analytical schema

The Phase 4.3.2 base route-month dataframe is intended to contain:

- `UNIQUE_CARRIER`
- `AIRLINE_ID`
- `UNIQUE_CARRIER_NAME`
- `YEAR`
- `MONTH`
- `YEAR_MONTH`
- `ORIGIN`
- `DEST`
- `PASSENGERS`
- `SEATS`
- `DEPARTURES_SCHEDULED`
- `DEPARTURES_PERFORMED`
- `DISTANCE`

The target analytical grain is exactly one row per:

`UNIQUE_CARRIER + YEAR + MONTH + ORIGIN + DEST`

No zero-activity filtering is applied in this phase.

In [50]:
route_month_df = (
    wn_f_historical[[
    'AIRLINE_ID',
    'UNIQUE_CARRIER_NAME',    
    'UNIQUE_CARRIER',
    'YEAR',
    'MONTH',
    'ORIGIN',
    'DEST',
    'PASSENGERS',
    'SEATS',
    'DEPARTURES_SCHEDULED',
    'DEPARTURES_PERFORMED',
    'DISTANCE'
]]
    .groupby(
        [
            'UNIQUE_CARRIER',
            'YEAR',
            'MONTH',
            'ORIGIN',
            'DEST'
        ],
        as_index=False
    )
    .agg(
        AIRLINE_ID=('AIRLINE_ID', 'first'),
        UNIQUE_CARRIER_NAME=('UNIQUE_CARRIER_NAME', 'first'),
        PASSENGERS=('PASSENGERS', 'sum'),
        SEATS=('SEATS', 'sum'),
        DEPARTURES_SCHEDULED=('DEPARTURES_SCHEDULED', 'sum'),
        DEPARTURES_PERFORMED=('DEPARTURES_PERFORMED', 'sum'),
        DISTANCE=('DISTANCE', 'first')
    )
)
print('COLUMNS IN route_month_df:', route_month_df.columns)
print('SHAPE OF route_month_df:', route_month_df.shape)

COLUMNS IN route_month_df: Index(['UNIQUE_CARRIER', 'YEAR', 'MONTH', 'ORIGIN', 'DEST', 'AIRLINE_ID',
       'UNIQUE_CARRIER_NAME', 'PASSENGERS', 'SEATS', 'DEPARTURES_SCHEDULED',
       'DEPARTURES_PERFORMED', 'DISTANCE'],
      dtype='str')
SHAPE OF route_month_df: (55395, 12)


In [51]:
route_month_df['YEAR_MONTH'] = pd.PeriodIndex.from_fields(
    year=route_month_df['YEAR'],
    month=route_month_df['MONTH'],
    freq='M'
)

route_month_df[
    ['YEAR_MONTH', 'YEAR', 'MONTH']
].head()

,YEAR_MONTH,YEAR,MONTH
0,2024-01,2024,1
1,2024-01,2024,1
2,2024-01,2024,1
3,2024-01,2024,1
4,2024-01,2024,1


In [52]:
route_month_df.duplicated(
    subset=[
        'UNIQUE_CARRIER',
        'YEAR',
        'MONTH',
        'ORIGIN',
        'DEST'
    ]
).sum()

np.int64(0)

In [53]:
additive_columns = [
    'PASSENGERS',
    'SEATS',
    'DEPARTURES_SCHEDULED',
    'DEPARTURES_PERFORMED'
]
pd.DataFrame({
    'RAW_TOTAL': wn_f_historical[additive_columns].sum(),
    'AGGREGATED_TOTAL': route_month_df[additive_columns].sum(),
    'DIFFERENCE': (
        wn_f_historical[additive_columns].sum()
        - route_month_df[additive_columns].sum()
    )
})

,RAW_TOTAL,AGGREGATED_TOTAL,DIFFERENCE
PASSENGERS,414121175.0,414121175.0,0.0
SEATS,550463235.0,550463235.0,0.0
DEPARTURES_SCHEDULED,3454479.0,3454479.0,0.0
DEPARTURES_PERFORMED,3433647.0,3433647.0,0.0


In [106]:
numeric_checks = [
    'PASSENGERS',
    'SEATS',
    'DEPARTURES_SCHEDULED',
    'DEPARTURES_PERFORMED',
    'DISTANCE'
]

print("Number of missing values in each numeric column:")
print(route_month_df.isna().sum())

print("Number of negative values in each numeric column:")
print((route_month_df[numeric_checks] < 0).sum())

Number of missing values in each numeric column:
UNIQUE_CARRIER          0
AIRLINE_ID              0
UNIQUE_CARRIER_NAME     0
YEAR                    0
MONTH                   0
YEAR_MONTH              0
ORIGIN                  0
DEST                    0
PASSENGERS              0
SEATS                   0
DEPARTURES_SCHEDULED    0
DEPARTURES_PERFORMED    0
DISTANCE                0
IS_ACTIVE               0
dtype: int64
Number of negative values in each numeric column:
PASSENGERS              0
SEATS                   0
DEPARTURES_SCHEDULED    0
DEPARTURES_PERFORMED    0
DISTANCE                0
dtype: int64


In [56]:
route_month_df['YEAR_MONTH'].agg(['min', 'max', 'nunique'])

min        2024-01
max        2026-05
nunique         29
Name: YEAR_MONTH, dtype: object

In [57]:
route_month_df[
    ['ORIGIN', 'DEST']
].drop_duplicates().shape[0]

4271

In [58]:
route_month_df.duplicated(
    subset=[
        'UNIQUE_CARRIER',
        'YEAR_MONTH',
        'ORIGIN',
        'DEST'
    ]
).sum()

np.int64(0)

In [59]:
route_month_df = route_month_df[
    [
        'UNIQUE_CARRIER',
        'AIRLINE_ID',
        'UNIQUE_CARRIER_NAME',
        'YEAR',
        'MONTH',
        'YEAR_MONTH',
        'ORIGIN',
        'DEST',
        'PASSENGERS',
        'SEATS',
        'DEPARTURES_SCHEDULED',
        'DEPARTURES_PERFORMED',
        'DISTANCE'
    ]
]
print(route_month_df.shape)
print(route_month_df.columns)

(55395, 13)
Index(['UNIQUE_CARRIER', 'AIRLINE_ID', 'UNIQUE_CARRIER_NAME', 'YEAR', 'MONTH',
       'YEAR_MONTH', 'ORIGIN', 'DEST', 'PASSENGERS', 'SEATS',
       'DEPARTURES_SCHEDULED', 'DEPARTURES_PERFORMED', 'DISTANCE'],
      dtype='str')


### Route-month aggregation validation

The WN `CLASS = "F"` source records were aggregated to exactly one observation per directional route-month.

#### Final output

- Raw WN `CLASS = "F"` rows: **139,082**
- Aggregated route-month rows: **55,395**
- Final analytical columns: **13**
- Unique directional routes: **4,271**
- Historical coverage: **2024-01 through 2026-05**
- Unique calendar months: **29**

The analytical key is:

`UNIQUE_CARRIER + YEAR + MONTH + ORIGIN + DEST`

An equivalent key using `YEAR_MONTH` is:

`UNIQUE_CARRIER + YEAR_MONTH + ORIGIN + DEST`

Both key representations produced **0 duplicate rows** after aggregation.

#### Additive-measure reconciliation

The following measures were summed across legitimate raw component records:

- `PASSENGERS`
- `SEATS`
- `DEPARTURES_SCHEDULED`
- `DEPARTURES_PERFORMED`

Raw and aggregated totals matched exactly for all four measures:

- `PASSENGERS`: 414,121,175 → 414,121,175
- `SEATS`: 550,463,235 → 550,463,235
- `DEPARTURES_SCHEDULED`: 3,454,479 → 3,454,479
- `DEPARTURES_PERFORMED`: 3,433,647 → 3,433,647

Difference for every additive measure: **0**

This confirms that aggregation neither lost nor duplicated reported operational activity.

#### Distance preservation

`DISTANCE` was validated before aggregation by checking the number of distinct distance values within every directional route-month.

No route-month contained conflicting distance values.

Therefore, one validated `DISTANCE` value is preserved for each aggregated route-month.

#### Data-quality checks

The final route-month schema contains:

- no missing values
- no negative values in `PASSENGERS`, `SEATS`, `DEPARTURES_SCHEDULED`, `DEPARTURES_PERFORMED`, or `DISTANCE`

#### Phase boundary

No zero-activity observations were removed during Phase 4.3.2.

Route-months containing zero passengers, zero seats, zero departures, or mixed activity states remain preserved for investigation in Phase 4.3.3.

Phase 4.3.2 does not define the final active route-month rule or route eligibility.

## Phase 4.3.3 — Zero-Activity Rules & Final Active Route-Month Definition

In [60]:
print('Number of rows with 0 passengers:',(route_month_df["PASSENGERS"] == 0).sum())
print('Number of rows with more than 0 passengers:',(route_month_df["PASSENGERS"] > 0).sum())

Number of rows with 0 passengers: 138
Number of rows with more than 0 passengers: 55257


In [61]:
print('Number of rows with 0 seats:',(route_month_df["SEATS"] == 0).sum())
print('Number of rows with more than 0 seats:',(route_month_df["SEATS"] > 0).sum())

Number of rows with 0 seats: 19
Number of rows with more than 0 seats: 55376


In [62]:
print('Number of rows with 0 departures scheduled:',(route_month_df["DEPARTURES_SCHEDULED"] == 0).sum())
print('Number of rows with more than 0 departures scheduled:',(route_month_df["DEPARTURES_SCHEDULED"] > 0).sum())

Number of rows with 0 departures scheduled: 4562
Number of rows with more than 0 departures scheduled: 50833


In [63]:
print('Number of rows with 0 departures performed:',(route_month_df["DEPARTURES_PERFORMED"] == 0).sum())
print('Number of rows with more than 0 departures performed:',(route_month_df["DEPARTURES_PERFORMED"] > 0).sum())

Number of rows with 0 departures performed: 19
Number of rows with more than 0 departures performed: 55376


In [64]:
zero_passenger_months = route_month_df.loc[
    route_month_df["PASSENGERS"] == 0
]
print(
    "Zero-passenger rows with positive seats:",
    (zero_passenger_months["SEATS"] > 0).sum()
)

print(
    "Zero-passenger rows with scheduled departures:",
    (zero_passenger_months["DEPARTURES_SCHEDULED"] > 0).sum()
)

print(
    "Zero-passenger rows with performed departures:",
    (zero_passenger_months["DEPARTURES_PERFORMED"] > 0).sum()
)

print(
    "Complete-zero route-months:",
    (
        (zero_passenger_months["SEATS"] == 0)
        & (zero_passenger_months["DEPARTURES_SCHEDULED"] == 0)
        & (zero_passenger_months["DEPARTURES_PERFORMED"] == 0)
    ).sum()
)

Zero-passenger rows with positive seats: 119
Zero-passenger rows with scheduled departures: 20
Zero-passenger rows with performed departures: 119
Complete-zero route-months: 0


In [65]:
zero_passenger_seats = route_month_df.loc[
    (route_month_df["PASSENGERS"] == 0) & (route_month_df["SEATS"] == 0),
    ["YEAR_MONTH",
    "ORIGIN",
    "DEST",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"]
]
print('departures performed counts for zero-passenger, zero-seat rows:', zero_passenger_seats['DEPARTURES_PERFORMED'].value_counts())
print('departures scheduled counts for zero-passenger, zero-seat rows:', zero_passenger_seats['DEPARTURES_SCHEDULED'].value_counts())

departures performed counts for zero-passenger, zero-seat rows: DEPARTURES_PERFORMED
0.0    19
Name: count, dtype: int64
departures scheduled counts for zero-passenger, zero-seat rows: DEPARTURES_SCHEDULED
1.0    11
2.0     6
3.0     2
Name: count, dtype: int64


In [66]:
positive_passenger_0seats = route_month_df.loc[
    (route_month_df["PASSENGERS"] > 0) & (route_month_df["SEATS"] == 0),
    ["YEAR_MONTH",
    "ORIGIN",
    "DEST",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"]
]
print('departures performed counts for positive-passenger, zero-seat rows:', positive_passenger_0seats['DEPARTURES_PERFORMED'].value_counts())
print('departures scheduled counts for positive-passenger, zero-seat rows:', positive_passenger_0seats['DEPARTURES_SCHEDULED'].value_counts())

departures performed counts for positive-passenger, zero-seat rows: Series([], Name: count, dtype: int64)
departures scheduled counts for positive-passenger, zero-seat rows: Series([], Name: count, dtype: int64)


In [67]:
positive_passenger_0sched = route_month_df.loc[
    (route_month_df["PASSENGERS"] > 0) & (route_month_df["DEPARTURES_SCHEDULED"] == 0),
    ["YEAR_MONTH",
    "ORIGIN",
    "DEST",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"]
]
print('departures performed counts for positive-passenger, zero-departures scheduled rows:', positive_passenger_0sched['DEPARTURES_PERFORMED'].value_counts())
print('departures scheduled counts for positive-passenger, zero-departures scheduled rows:', positive_passenger_0sched['DEPARTURES_SCHEDULED'].value_counts())

departures performed counts for positive-passenger, zero-departures scheduled rows: DEPARTURES_PERFORMED
1.0     3795
2.0      405
3.0      107
5.0       38
4.0       36
6.0       22
7.0       10
11.0       7
8.0        5
10.0       5
9.0        4
16.0       4
18.0       2
15.0       1
28.0       1
13.0       1
12.0       1
Name: count, dtype: int64
departures scheduled counts for positive-passenger, zero-departures scheduled rows: DEPARTURES_SCHEDULED
0.0    4444
Name: count, dtype: int64


In [68]:

positive_passenger_0performed = route_month_df.loc[
    (route_month_df["PASSENGERS"] > 0) & (route_month_df["DEPARTURES_PERFORMED"] == 0),
    ["YEAR_MONTH",
    "ORIGIN",
    "DEST",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"]
]
print('departures performed counts for positive-passenger, zero-departures performed rows:', positive_passenger_0performed['DEPARTURES_PERFORMED'].value_counts())
print('departures scheduled counts for positive-passenger, zero-departures performed rows:', positive_passenger_0performed['DEPARTURES_SCHEDULED'].value_counts())

departures performed counts for positive-passenger, zero-departures performed rows: Series([], Name: count, dtype: int64)
departures scheduled counts for positive-passenger, zero-departures performed rows: Series([], Name: count, dtype: int64)


In [69]:
compare_perf_gt_sched = route_month_df.loc[
    (route_month_df["DEPARTURES_PERFORMED"] > route_month_df["DEPARTURES_SCHEDULED"]) ,
    ["YEAR_MONTH",
    "ORIGIN",
    "DEST",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"]
]
print(len(compare_perf_gt_sched), ' rows where departures performed is greater than departures scheduled')
compare_perf_gt_sched.head(10)

8300  rows where departures performed is greater than departures scheduled


,YEAR_MONTH,ORIGIN,DEST,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
15,2024-01,ABQ,TUL,1.0,175.0,0.0,1.0
16,2024-01,ALB,BOS,51.0,175.0,0.0,1.0
22,2024-01,ALB,PIT,23.0,175.0,0.0,1.0
25,2024-01,AMA,BWI,2.0,143.0,0.0,1.0
30,2024-01,AMA,STL,173.0,175.0,0.0,1.0
32,2024-01,ATL,BDL,90.0,175.0,0.0,1.0
46,2024-01,ATL,ISP,134.0,143.0,0.0,1.0
58,2024-01,ATL,MSY,9406.0,13433.0,84.0,87.0
63,2024-01,ATL,PHX,4917.0,5741.0,34.0,35.0
64,2024-01,ATL,PIT,3867.0,7694.0,49.0,50.0


In [70]:
pd.crosstab(
    route_month_df["PASSENGERS"] > 0,
    route_month_df["DEPARTURES_PERFORMED"] > 0,
    rownames=["PASSENGERS > 0"],
    colnames=["DEPARTURES_PERFORMED > 0"]
)

DEPARTURES_PERFORMED > 0,False,True
PASSENGERS > 0,,
False,19,119
True,0,55257


In [71]:
nonperformed_route_months = route_month_df.loc[
    route_month_df["DEPARTURES_PERFORMED"] == 0,
    [
        "YEAR_MONTH",
        "ORIGIN",
        "DEST",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED"
    ]
]
print("Non-performed route-months:", len(nonperformed_route_months))

print(
    "Directional routes affected:",
    nonperformed_route_months[["ORIGIN", "DEST"]]
    .drop_duplicates()
    .shape[0]
)

nonperformed_route_months.sort_values(
    ["ORIGIN", "DEST", "YEAR_MONTH"]
)

Non-performed route-months: 19
Directional routes affected: 19


,YEAR_MONTH,ORIGIN,DEST,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
45910,2026-01,AUS,SJU,0.0,0.0,1.0,0.0
19350,2024-11,CMH,SRQ,0.0,0.0,2.0,0.0
51980,2026-04,CUN,SAT,0.0,0.0,1.0,0.0
28623,2025-04,CVG,TPA,0.0,0.0,1.0,0.0
481,2024-01,DEN,BDL,0.0,0.0,1.0,0.0
42637,2025-11,FLL,MBJ,0.0,0.0,1.0,0.0
19717,2024-11,HOU,SRQ,0.0,0.0,2.0,0.0
19752,2024-11,IND,SRQ,0.0,0.0,3.0,0.0
42983,2025-11,MBJ,FLL,0.0,0.0,1.0,0.0
23595,2025-01,MBJ,MCI,0.0,0.0,1.0,0.0


In [72]:
pd.crosstab(
    route_month_df["SEATS"] > 0,
    route_month_df["DEPARTURES_PERFORMED"] > 0,
    rownames=["SEATS > 0"],
    colnames=["DEPARTURES_PERFORMED > 0"]
)

DEPARTURES_PERFORMED > 0,False,True
SEATS > 0,,
False,19,0
True,0,55376


In [73]:
route_month_df["IS_ACTIVE"] = (
    route_month_df["DEPARTURES_PERFORMED"] > 0
)
route_month_df["IS_ACTIVE"].value_counts()

IS_ACTIVE
True     55376
False       19
Name: count, dtype: int64

In [74]:
old_active = route_month_df["PASSENGERS"] > 0
new_active = route_month_df["IS_ACTIVE"]

print(
    "Route-month classifications changed:",
    (old_active != new_active).sum()
)

Route-month classifications changed: 119


In [75]:
forecast_origin = pd.Period("2025-01", freq="M")

historical_active = route_month_df.loc[
    (route_month_df["YEAR_MONTH"] < forecast_origin)
    & (route_month_df["IS_ACTIVE"]),
    ["ORIGIN", "DEST", "YEAR_MONTH"]
]

In [76]:
historical_active_counts = (
    historical_active
    .groupby(["ORIGIN", "DEST"])
    .size()
    .reset_index(name="HISTORICAL_ACTIVE_MONTHS")
)

In [77]:
recent_active_routes = (
    historical_active.loc[
        historical_active["YEAR_MONTH"] >= forecast_origin - 3,
        ["ORIGIN", "DEST"]
    ]
    .drop_duplicates()
)
eligible_2025_01 = (
    historical_active_counts.loc[
        historical_active_counts["HISTORICAL_ACTIVE_MONTHS"] >= 6
    ]
    .merge(
        recent_active_routes,
        on=["ORIGIN", "DEST"],
        how="inner"
    )
)

print("Eligible routes at 2025-01:", len(eligible_2025_01))

Eligible routes at 2025-01: 1739


In [78]:
forecast_origin = pd.Period("2026-01", freq="M")

historical_active = route_month_df.loc[
    (route_month_df["YEAR_MONTH"] < forecast_origin)
    & (route_month_df["IS_ACTIVE"]),
    ["ORIGIN", "DEST", "YEAR_MONTH"]
]

historical_active_counts = (
    historical_active
    .groupby(["ORIGIN", "DEST"])
    .size()
    .reset_index(name="HISTORICAL_ACTIVE_MONTHS")
)

recent_active_routes = (
    historical_active.loc[
        historical_active["YEAR_MONTH"] >= forecast_origin - 3,
        ["ORIGIN", "DEST"]
    ]
    .drop_duplicates()
)

eligible_2026_01 = (
    historical_active_counts.loc[
        historical_active_counts["HISTORICAL_ACTIVE_MONTHS"] >= 6
    ]
    .merge(
        recent_active_routes,
        on=["ORIGIN", "DEST"],
        how="inner"
    )
)

print("Eligible routes at 2026-01:", len(eligible_2026_01))

Eligible routes at 2026-01: 1854


In [79]:
forecast_origin = pd.Period("2026-01", freq="M")

old_historical_active = route_month_df.loc[
    (route_month_df["YEAR_MONTH"] < forecast_origin)
    & (route_month_df["PASSENGERS"] > 0),
    ["ORIGIN", "DEST", "YEAR_MONTH"]
]

old_historical_active_counts = (
    old_historical_active
    .groupby(["ORIGIN", "DEST"])
    .size()
    .reset_index(name="HISTORICAL_ACTIVE_MONTHS")
)

old_recent_active_routes = (
    old_historical_active.loc[
        old_historical_active["YEAR_MONTH"] >= forecast_origin - 3,
        ["ORIGIN", "DEST"]
    ]
    .drop_duplicates()
)

old_eligible_2026_01 = (
    old_historical_active_counts.loc[
        old_historical_active_counts["HISTORICAL_ACTIVE_MONTHS"] >= 6
    ]
    .merge(
        old_recent_active_routes,
        on=["ORIGIN", "DEST"],
        how="inner"
    )
)

print("Old definition:", len(old_eligible_2026_01))
print("New definition:", len(eligible_2026_01))

Old definition: 1851
New definition: 1854


In [80]:
new_only_2026 = (
    eligible_2026_01[
        ["ORIGIN", "DEST"]
    ]
    .merge(
        old_eligible_2026_01[
            ["ORIGIN", "DEST"]
        ],
        on=["ORIGIN", "DEST"],
        how="left",
        indicator=True
    )
    .loc[
        lambda df: df["_merge"] == "left_only",
        ["ORIGIN", "DEST"]
    ]
)

new_only_2026

,ORIGIN,DEST
1007,MCO,BOS
1621,SJC,OAK
1755,STL,IND


In [81]:
changed_months_new_routes = route_month_df.loc[
    (route_month_df["YEAR_MONTH"] < pd.Period("2026-01", freq="M"))
    & (
        (
            (route_month_df["ORIGIN"] == "MCO")
            & (route_month_df["DEST"] == "BOS")
        )
        | (
            (route_month_df["ORIGIN"] == "SJC")
            & (route_month_df["DEST"] == "OAK")
        )
        | (
            (route_month_df["ORIGIN"] == "STL")
            & (route_month_df["DEST"] == "IND")
        )
    )
    & (route_month_df["PASSENGERS"] == 0)
    & (route_month_df["DEPARTURES_PERFORMED"] > 0),
    [
        "YEAR_MONTH",
        "ORIGIN",
        "DEST",
        "PASSENGERS",
        "SEATS",
        "DEPARTURES_SCHEDULED",
        "DEPARTURES_PERFORMED"
    ]
].sort_values(
    ["ORIGIN", "DEST", "YEAR_MONTH"]
)

changed_months_new_routes

,YEAR_MONTH,ORIGIN,DEST,PASSENGERS,SEATS,DEPARTURES_SCHEDULED,DEPARTURES_PERFORMED
35329,2025-07,MCO,BOS,0.0,175.0,0.0,1.0
18785,2024-10,SJC,OAK,0.0,143.0,0.0,1.0
15438,2024-08,STL,IND,0.0,143.0,0.0,1.0


In [82]:
three_routes_history = route_month_df.loc[
    (route_month_df["YEAR_MONTH"] < pd.Period("2026-01", freq="M"))
    & (
        route_month_df[["ORIGIN", "DEST"]]
        .apply(tuple, axis=1)
        .isin([
            ("MCO", "BOS"),
            ("SJC", "OAK"),
            ("STL", "IND")
        ])
    )
].copy()

three_routes_history["OLD_ACTIVE"] = (
    three_routes_history["PASSENGERS"] > 0
)

three_routes_history["NEW_ACTIVE"] = (
    three_routes_history["DEPARTURES_PERFORMED"] > 0
)

three_routes_history.groupby(
    ["ORIGIN", "DEST"]
)[["OLD_ACTIVE", "NEW_ACTIVE"]].sum()

,,OLD_ACTIVE,NEW_ACTIVE
ORIGIN,DEST,,
MCO,BOS,5,6
SJC,OAK,5,6
STL,IND,5,6


### Activity-State Audit and Final Definition

The aggregated WN route-month foundation contains 55,395 source-observed route-months.

Marginal activity counts:

- `PASSENGERS == 0`: 138
- `PASSENGERS > 0`: 55,257
- `SEATS == 0`: 19
- `SEATS > 0`: 55,376
- `DEPARTURES_SCHEDULED == 0`: 4,562
- `DEPARTURES_SCHEDULED > 0`: 50,833
- `DEPARTURES_PERFORMED == 0`: 19
- `DEPARTURES_PERFORMED > 0`: 55,376

Important observed combinations:

- 119 route-months had zero passengers but positive seats and positive performed departures.
- 19 route-months had zero passengers, zero seats, and zero performed departures, but positive scheduled departures.
- 0 route-months had positive passengers with zero seats.
- 0 route-months had positive passengers with zero performed departures.
- 4,444 route-months had positive passengers with zero scheduled departures.
- 4,562 route-months had performed departures despite zero scheduled departures.
- 19 route-months had scheduled departures but zero performed departures.
- 8,300 route-months had performed departures greater than scheduled departures.
- 0 route-months were complete-zero observations across passengers, seats, scheduled departures, and performed departures.

`SEATS > 0` and `DEPARTURES_PERFORMED > 0` classified all 55,395 route-months identically in this dataset. However, `DEPARTURES_PERFORMED` is the more direct operational measure because it explicitly represents flights that were actually performed.

`DEPARTURES_SCHEDULED > 0` is not suitable as the operational activity definition because many route-months had performed service despite zero scheduled departures, while 19 route-months had scheduled departures but no performed operation.

#### Final definitions

A **source-observed route-month** is a directional WN route-month present in the aggregated BTS `CLASS = F` source.

An **operationally active route-month** is defined as:

`DEPARTURES_PERFORMED > 0`

A **zero-passenger active route-month** is therefore valid when `PASSENGERS == 0` but `DEPARTURES_PERFORMED > 0`. There are 119 such route-months, and they are retained as legitimate observed zero-passenger outcomes.

A **source-observed inactive route-month** is a source-observed route-month where `DEPARTURES_PERFORMED == 0`. There are 19 such observations. They should remain in the source-observed foundation with their activity status preserved, but they should not count as active months in the later route-eligibility calculation.

The previous Phase 4.2 exploratory definition, `PASSENGERS > 0`, differs from the final operational definition for 119 route-months. All 119 are zero-passenger months with performed operation.

#### Eligibility validation

Using the approved leakage-safe rule:

- at least 6 historical active months before the forecast origin
- active at least once during the previous 3 calendar months

the results are:

- `2025-01`: old definition = 1,739; final definition = 1,739
- `2026-01`: old definition = 1,851; final definition = 1,854

The three additional routes at `2026-01` are:

- MCO → BOS
- SJC → OAK
- STL → IND

Each route previously had 5 historical active months and gained a sixth historical active month because a zero-passenger month with a performed departure is now correctly recognized as operationally active. Each route already satisfied the 3-month recency condition.

The small eligibility change does not provide evidence that the approved 6-month historical threshold or 3-month recency window needs to be re-selected.

All eligibility calculations used only route activity strictly before the forecast origin.

## Phase 4.3.4 — Reusable Processing Pipeline & Eligibility Integration

In [83]:
from pathlib import Path
import sys

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.data_processing as dp

In [84]:
reusable1_route_month_df = dp.build_route_month_dataset(df_historical)
print("reusable1_route_month_df shape:", reusable1_route_month_df.shape)
print("reusable1_route_month_df columns:", reusable1_route_month_df.columns.tolist())

reusable1_route_month_df shape: (55395, 14)
reusable1_route_month_df columns: ['UNIQUE_CARRIER', 'AIRLINE_ID', 'UNIQUE_CARRIER_NAME', 'YEAR', 'MONTH', 'YEAR_MONTH', 'ORIGIN', 'DEST', 'PASSENGERS', 'SEATS', 'DEPARTURES_SCHEDULED', 'DEPARTURES_PERFORMED', 'DISTANCE', 'IS_ACTIVE']


In [85]:
print("YEAR_MONTH dtype:", reusable1_route_month_df["YEAR_MONTH"].dtype)

print(
    reusable1_route_month_df["IS_ACTIVE"].value_counts()
)

YEAR_MONTH dtype: period[M]
IS_ACTIVE
True     55376
False       19
Name: count, dtype: int64


In [86]:
zero_passenger_active = reusable1_route_month_df[
    (reusable1_route_month_df["PASSENGERS"] == 0) &
    (reusable1_route_month_df["IS_ACTIVE"])
]

print(
    "Zero-passenger active route-months:",
    len(zero_passenger_active)
)

Zero-passenger active route-months: 119


In [87]:
reusable1_route_month_df.duplicated(
    subset=["UNIQUE_CARRIER", "YEAR", "MONTH", "ORIGIN", "DEST"]
).sum()

np.int64(0)

In [88]:

reusable1_route_month_df = dp.build_route_month_dataset(df_historical)

print(reusable1_route_month_df.shape)

(55395, 14)


In [89]:

reusable1_route_month_df = dp.build_route_month_dataset(df_historical)

print(reusable1_route_month_df.shape)

reusable1_route_month_df[[
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"
]].sum()

(55395, 14)


PASSENGERS              414121175.0
SEATS                   550463235.0
DEPARTURES_SCHEDULED      3454479.0
DEPARTURES_PERFORMED      3433647.0
dtype: float64

In [90]:
history_eligible = dp.get_eligible_routes(
    reusable1_route_month_df,
    "2026-01"
)

print(history_eligible.shape)

print(
    history_eligible["HISTORICAL_ACTIVE_MONTHS"].min()
)

(1854, 3)
6


In [91]:
dp.get_eligible_routes(
    reusable1_route_month_df,
    "2026-01"
)


,ORIGIN,DEST,HISTORICAL_ACTIVE_MONTHS
0,ABQ,AUS,24
1,ABQ,BNA,12
2,ABQ,BUR,24
3,ABQ,BWI,24
4,ABQ,DAL,24
...,...,...,...
1849,VPS,BNA,24
1850,VPS,BWI,17
1851,VPS,DAL,24
1852,VPS,HOU,24


In [92]:
eligible_2026_01 = dp.get_eligible_routes(
    reusable1_route_month_df,
    "2026-01"
)

print(eligible_2026_01.shape)

(1854, 3)


In [93]:
eligible_2025_01 = dp.get_eligible_routes(
    reusable1_route_month_df,
    "2025-01"
)

print(eligible_2025_01.shape)

(1739, 3)


In [94]:
routes_to_check = pd.DataFrame({
    "ORIGIN": ["MCO", "SJC", "STL"],
    "DEST": ["BOS", "OAK", "IND"],
})

routes_to_check.merge(
    eligible_2026_01,
    on=["ORIGIN", "DEST"],
    how="left",
    indicator=True
)

,ORIGIN,DEST,HISTORICAL_ACTIVE_MONTHS,_merge
0,MCO,BOS,6,both
1,SJC,OAK,6,both
2,STL,IND,6,both


In [95]:
eligible_2025_01 = dp.get_eligible_routes(
    reusable1_route_month_df,
    "2025-01"
)

eligible_2026_01 = dp.get_eligible_routes(
    reusable1_route_month_df,
    "2026-01"
)

print(len(eligible_2025_01))
print(len(eligible_2026_01))

1739
1854


In [96]:
print("Shape:", reusable1_route_month_df.shape)

print(
    "Directional routes:",
    reusable1_route_month_df[
        ["ORIGIN", "DEST"]
    ].drop_duplicates().shape[0]
)

print(
    "Months:",
    reusable1_route_month_df["YEAR_MONTH"].nunique()
)

print(
    "Min month:",
    reusable1_route_month_df["YEAR_MONTH"].min()
)

print(
    "Max month:",
    reusable1_route_month_df["YEAR_MONTH"].max()
)

print(
    "Duplicate keys:",
    reusable1_route_month_df.duplicated(
        subset=[
            "UNIQUE_CARRIER",
            "YEAR",
            "MONTH",
            "ORIGIN",
            "DEST",
        ]
    ).sum()
)

print(
    "Active:",
    reusable1_route_month_df["IS_ACTIVE"].sum()
)

print(
    "Inactive:",
    (~reusable1_route_month_df["IS_ACTIVE"]).sum()
)

print(
    "Zero-passenger active:",
    (
        (reusable1_route_month_df["PASSENGERS"] == 0) &
        (reusable1_route_month_df["IS_ACTIVE"])
    ).sum()
)

Shape: (55395, 14)
Directional routes: 4271
Months: 29
Min month: 2024-01
Max month: 2026-05
Duplicate keys: 0
Active: 55376
Inactive: 19
Zero-passenger active: 119


In [97]:
comparison_columns = [
    "UNIQUE_CARRIER",
    "AIRLINE_ID",
    "UNIQUE_CARRIER_NAME",
    "YEAR",
    "MONTH",
    "YEAR_MONTH",
    "ORIGIN",
    "DEST",
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED",
    "DISTANCE",
    "IS_ACTIVE",
]

approved_sorted = (
    route_month_df[comparison_columns]
    .sort_values(
        ["UNIQUE_CARRIER", "YEAR", "MONTH", "ORIGIN", "DEST"]
    )
    .reset_index(drop=True)
)

reusable_sorted = (
    reusable1_route_month_df[comparison_columns]
    .sort_values(
        ["UNIQUE_CARRIER", "YEAR", "MONTH", "ORIGIN", "DEST"]
    )
    .reset_index(drop=True)
)

In [98]:
print("Same shape:", approved_sorted.shape == reusable_sorted.shape)

print(
    "Same columns:",
    approved_sorted.columns.tolist()
    == reusable_sorted.columns.tolist()
)

print(
    "Full dataframe match:",
    approved_sorted.equals(reusable_sorted)
)

Same shape: True
Same columns: True
Full dataframe match: True


In [99]:
processed_df = reusable1_route_month_df.copy()

processed_df["YEAR_MONTH"] = (
    processed_df["YEAR_MONTH"].astype(str)
)

In [100]:
processed_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "wn_route_month.csv"
)

processed_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

processed_df.to_csv(
    processed_path,
    index=False
)

print(processed_path)

c:\Users\PC\airline-demand-capacity-planning\data\processed\wn_route_month.csv


In [107]:
reloaded_processed_df = pd.read_csv(processed_path)

print("Shape:", reloaded_processed_df.shape)
print("Columns:", reloaded_processed_df.columns.tolist())
print("YEAR_MONTH dtype:", reloaded_processed_df["YEAR_MONTH"].dtype)
print("IS_ACTIVE dtype:", reloaded_processed_df["IS_ACTIVE"].dtype)
print("Min month:", reloaded_processed_df["YEAR_MONTH"].min())
print("Max month:", reloaded_processed_df["YEAR_MONTH"].max())

reloaded_processed_df[[
    "PASSENGERS",
    "SEATS",
    "DEPARTURES_SCHEDULED",
    "DEPARTURES_PERFORMED"
]].sum()

Shape: (55395, 14)
Columns: ['UNIQUE_CARRIER', 'AIRLINE_ID', 'UNIQUE_CARRIER_NAME', 'YEAR', 'MONTH', 'YEAR_MONTH', 'ORIGIN', 'DEST', 'PASSENGERS', 'SEATS', 'DEPARTURES_SCHEDULED', 'DEPARTURES_PERFORMED', 'DISTANCE', 'IS_ACTIVE']
YEAR_MONTH dtype: str
IS_ACTIVE dtype: bool
Min month: 2024-01
Max month: 2026-05


PASSENGERS              414121175.0
SEATS                   550463235.0
DEPARTURES_SCHEDULED      3454479.0
DEPARTURES_PERFORMED      3433647.0
dtype: float64